In [12]:
import pathlib
import utils
import random
import csv

In [7]:
answers = [utils.read_csv_dict(path) for path in pathlib.Path("../data/results/lrm").glob("*_reasons_*.csv")]
reasonings = [utils.read_csv_dict(path) for path in pathlib.Path("../data/results/lrm").glob("*_responses_*.csv")]


inst_answers = [utils.read_csv_dict(path) for path in pathlib.Path("../data/results/lrm").glob("*_reasons_inst*.csv")]
inst_reasonings = [utils.read_csv_dict(path) for path in pathlib.Path("../data/results/lrm").glob("*_responses_inst*.csv")]

pref_answers = [utils.read_csv_dict(path) for path in pathlib.Path("../data/results/lrm").glob("*_reasons_pref*.csv")]
pref_reasonings = [utils.read_csv_dict(path) for path in pathlib.Path("../data/results/lrm").glob("*_responses_pref*.csv")]

temp_answers = [utils.read_csv_dict(path) for path in pathlib.Path("../data/results/lrm").glob("*_reasons_temp*.csv")]
temp_reasonings = [utils.read_csv_dict(path) for path in pathlib.Path("../data/results/lrm").glob("*_responses_temp*.csv")]

In [102]:
def get_answer(ans: str):
    if "</think>\\n\\n" in ans:
        think = ans.rfind("</think>\\n\\n") + len("</think>\\n\\n")
        ans = ans[think:]


    ans = ans.replace("*", "")
    ans = ans.replace("\\n", "")
    ans = ans.replace("'", "")
    ans = ans.replace("(", "")
    ans = ans.replace(")", "")
    ans = ans.replace("answer:", "")
    ans = ans.replace("the answer is", "")
    ans = ans.replace("\\text{", "")

    boxed = "boxed\\{"
    y = "Yes"
    n = "No"

    if boxed in ans:
        loc = ans.find(boxed)
        # return ans[loc + len(boxed):]
        if "}" in ans: 
             return ans[loc + len(boxed):ans.rfind("}")]
        return ans[loc + len(boxed):]
    
    altboxed = "boxed{"
    if altboxed in ans:
        loc = ans.find(altboxed)
        # return ans[loc + len(boxed):]
        if "}" in ans: 
             return ans[loc + len(altboxed):ans.rfind("}")]
        return ans[loc + len(altboxed):]
    
    altboxed = "\\text{"
    if altboxed in ans:
        loc = ans.find(altboxed)
        # return ans[loc + len(boxed):]
        if "}" in ans: 
             return ans[loc + len(altboxed):ans.rfind("}")]
        return ans[loc + len(altboxed):]


    ans = ans.lower()

    # not simple case
    temp = False
    temp_nonces = ["gextravaganza" , "blicketbash" , "wugfest" , "fepfestival" , "daxday"]
    bad_nonces = ["blicktash", "bicketbash"]
    for nonce in temp_nonces:
        if nonce in ans:
            temp = True
    
    
    if ans == "b":
        return "b"

    for nonce in bad_nonces:
        if nonce in ans:
            temp = True

    if not temp:
        if "yes" in ans[:len(y)]:
            return y
        
        if "no" in ans[:len(n)]:
            return n
        
        if "yes" in ans[-len(y) - 2:]:
            return y
        
        if "no" in ans[-len(n) - 2:]:
            return n
        
        if "yes," in ans or "yes." in ans:
            return y
        
        if "no," in ans or "no." in ans:
            return n
        
        if "neither" in ans:
            return "neither"
        return ans

    ans = ans.replace(".", "")
    if temp:
        for nonce in temp_nonces:
            if nonce in ans[-len(nonce):]:
                return nonce
            
        for nonce in temp_nonces:
            if nonce in ans[:len(nonce) - 2]:
                return nonce
            
        for nonce in bad_nonces:
            if nonce in ans[-len(nonce):]:
                return nonce
            
        for nonce in bad_nonces:
            if nonce in ans[:len(nonce) - 2]:
                return nonce

In [9]:
insts = [utils.read_csv_dict(path) for path in pathlib.Path("../data/stimuli-nonce/split_inst").glob("*.csv")]
prefs = [utils.read_csv_dict(path) for path in pathlib.Path("../data/stimuli-nonce/split_pref").glob("*.csv")]
temps = [utils.read_csv_dict(path) for path in pathlib.Path("../data/stimuli-nonce/split_temp").glob("*.csv")]

inst_labels = []
for inst_list in insts:
    for inst in inst_list:
        inst_labels.append(inst["label"])


pref_labels = []
for pref_list in prefs:
    for pref in pref_list:
        pref_labels.append(pref["label"])

temp_labels = []
for temp_list in temps:
    for temp in temp_list:
        temp_labels.append(temp["label"])
all_labels = []
all_labels.extend(inst_labels)
all_labels.extend(pref_labels)
all_labels.extend(temp_labels)


In [72]:
reasons = []
p_changed_response = utils.read_csv_dict("../data/results/lrm/pref-changeddeepseek-ai_DeepSeek-R1-Distill-Qwen-14B_responses_.csv")
it = iter(p_changed_response)
for pref_reason_list in pref_reasonings:
    for pref_reason in pref_reason_list:
        if "warm weather" not in pref_reason["idx"]:
            reasons.append(pref_reason["idx"])
        else:
            reasons.append(next(it)["idx"])


with open("pref_reasons.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(("reason", "0"))
    writer.writerows(zip(reasons, [0] * len(reasons)))

In [15]:
import csv


all_answers = []
for reason_list in reasonings:
    for reason in reason_list:
        all_answers.append(get_answer(reason["idx"]))




inst_answers = []
for inst_reason_list in inst_reasonings:
    for inst_reason in inst_reason_list:
        inst_answers.append(get_answer(inst_reason["idx"]))

pref_answers = []
for pref_reason_list in pref_reasonings:
    for pref_reason in pref_reason_list:
        pref_answers.append(get_answer(pref_reason["idx"]))


i = 0
temp_answers = []
for temp_reason_list in temp_reasonings:
    for temp_reason in temp_reason_list:
        ans = get_answer(temp_reason["idx"])
        if not ans is None:
            temp_answers.append(get_answer(temp_reason["idx"]))
        else:
            temp_answers.append(temp_reason["idx"])




with open("answers.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(("response", "label"))
    writer.writerows(zip(all_answers, all_labels))

with open("pref_answers.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(("response", "label"))
    writer.writerows(zip(pref_answers, pref_labels))

with open("inst_answers.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(("response", "label"))
    writer.writerows(zip(inst_answers, inst_labels))

with open("temp_answers.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(("response", "label"))
    writer.writerows(zip(temp_answers, temp_labels))

In [16]:

inst_answers = []
for inst_reason_list in inst_reasonings:
    for inst_reason in inst_reason_list:
        inst_answers.append(inst_reason["idx"])

with open("inst_reasons.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(("response", "label"))
    writer.writerows(zip(inst_answers, inst_labels))

In [17]:
temp_answers = []
for temp_reason_list in temp_reasonings:
    for temp_reason in temp_reason_list:
        temp_answers.append(temp_reason["idx"])
        
with open("temp_reasons.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(("response", "label"))
    writer.writerows(zip(temp_answers, temp_labels))

In [ ]:
# temp_nonces = ["gextravaganza" , "blicketbash" , "wugfest" , "fepfestival" , "daxday"]
# errs = [(ans, idx) for idx, ans in enumerate(temp_answers) if not str(ans).lower() in temp_nonces]
# errs
# # temp_answers

In [30]:
import csv
p_changed_reason = utils.read_csv_dict("../data/results/lrm/pref-changeddeepseek-ai_DeepSeek-R1-Distill-Qwen-14B_reasons_.csv")
p_changed_response = utils.read_csv_dict("../data/results/lrm/pref-changeddeepseek-ai_DeepSeek-R1-Distill-Qwen-14B_responses_.csv")

p_changed = []
for response in p_changed_response:
    p_changed.append(get_answer(response["idx"]))

print(p_changed)

# utils.write_dict_list_to_csv("lrm_pref_changed_answers.csv", p_changed)

with open("lrm_pref_changed_answers.csv", "w", newline="") as file:
    writer = csv.writer(file)
    # writer.writerow("response")

    writer.writerows(p_changed)

['No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'Yes', 'Yes', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'Yes', 'Yes', 'No', 'No', 'Yes', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'Yes', 'Yes', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'Yes', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'Yes', 'Yes', 'No', 'No', 'Yes', 'Yes', 'No', 'No', 'Yes', 'Yes', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'Yes', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'Yes', '

In [50]:
fixes = []
for answer_list in answers:
    individual_fixes = []
    for i, answer in enumerate(answer_list):
        if answer["answer"] == "ERROR":
            individual_fixes.append(i)
    fixes.append(individual_fixes)
fixes[0]

[2, 14, 17, 32, 40, 43, 45, 64, 66, 67, 68, 76, 80, 81, 86, 87, 89]

In [ ]:
for idx in range(len(fixes)):
    for i in fixes[idx]:
        think = reasonings[idx][i]["idx"].rfind("</think>\\n\\n") + len("</think>\\n\\n")
        end = reasonings[idx][i]["idx"][think:]

        # end = end.replace("\\n", " ")



        if get_answer(end) == "None" or get_answer(end) is None:
            nonce ="daxday"

            print(end[-len(nonce):])
            print(end[:len(nonce)])
            print(end[-len(nonce):])
            print(end[:len(nonce)])
            print("---------------")
            raise ValueError(idx, i, end)

rst.')
daxday
rst.')
daxd
---------------


ValueError: (24, 60, 'daxday\\n\\nStep-by-step explanation: The sentence structure indicates that daxday happened first, leading to blicketbash as a consequence. The word "consequently" shows cause and effect, so daxday started first.\')')

In [ ]:
# for idx in range(len(fixes)):
#     for i in fixes[idx]:
#         think = reasonings[idx][i]["idx"].rfind("</think>\\n\\n") + len("</think>\\n\\n")
        
#         end = reasonings[idx][i]["idx"][think:]
#         start = end
        

#         end = end.replace("*", "").lower()
#         end = end.replace("\\n", " ")
#         end = end.replace("'", "")
#         end = end.replace("(", "")
#         end = end.replace(")", "")
#         # end = end.replace(".", "")
#         end = end.replace("answer:", "")
#         end = end.replace("the answer is", "")
        
#         end = end.strip()

#         print(idx, i, get_answer(end))


        # do_print = False
        # # if not "answer: no" in end:
        # #     if  not "answer: yes" in end:
        # #         if not "yes" in end[:3]:
        # #             if not "no" in end[:2]:
        # #                 if not "the answer is no" in end.replace(":", " "):
        # #                     if not "the answer is yes" in end.replace(":", " "):
        # #                         do_print = False

        # if not "yes" in end[:3]:
        #     if not "no" in end[:2]:
        #         if not "yes" in end[-3 -2:]:
        #             if not "no" in end[-2 -2:]:
        #                 do_print = False

        # temp_nonces = ["gextravaganza" , "blicketbash" , "wugfest" , "fepfestival" , "daxday"]
        # for nonce in temp_nonces:
        #     if nonce in end:
        #         do_print = True

        # for nonce in temp_nonces:
        #     if nonce in end[:len(nonce)]:
        #         do_print = False
        
        # for nonce in temp_nonces:
        #     if nonce in end[-len(nonce)  - 2:]:
        #         do_print = False

        # for nonce in temp_nonces:
        #     if  "answer: " + nonce in end:
        #         do_print = False

        # for nonce in temp_nonces:
        #     if  "answer:" + nonce in end:
        #         print("answer:" + nonce)
        #         do_print = False
        

        # if do_print:
            # print(end)
                

0 2 No
0 14 No
0 17 No
0 32 No
0 40 No
0 43 No
0 45 No
0 64 No
0 66 Yes
0 67 No
0 68 No
0 76 Yes
0 80 No
0 81 No
0 86 No
0 87 No
0 89 No
1 0 No
1 2 No
1 3 No
1 5 No
1 6 No
1 12 No
1 15 No
1 22 No
1 29 No
1 30 Yes
1 31 No
1 32 No
1 34 No
1 40 No
1 44 No
1 60 No
1 61 No
1 67 No
1 68 No
1 84 No
1 86 No
1 88 No
1 90 Yes
1 91 No
1 97 Yes
2 3 No
2 10 Yes
2 11 Yes
2 13 Yes
2 24 No
2 27 No
2 33 No
2 42 No
2 47 Yes
2 48 No
2 50 Yes
2 51 Yes
2 56 Yes
2 59 Yes
2 60 No
2 61 No
2 66 Yes
2 75 Yes
2 76 No
2 77 Yes
2 80 No
2 81 No
2 86 No
2 95 Yes
3 3 No
3 4 Yes
3 7 No
3 12 No
3 13 No
3 18 No
3 34 No
3 40 Yes
3 46 No
3 49 No
3 61 No
3 64 No
3 68 No
3 69 No
3 80 No
3 81 No
3 86 No
3 93 Yes
4 2 No
4 10 No
4 12 No
4 16 No
4 27 No
4 41 No
4 52 No
4 66 No
4 73 Yes
4 82 No
5 2 No
5 5 No
5 22 No
5 32 No
5 33 Yes
5 36 No
5 39 No
5 46 No
5 49 No
5 50 Yes
5 61 No
5 63 Yes
5 65 No
5 75 Yes
5 82 No
5 83 No
5 91 Yes
6 0 No
6 4 No
6 9 No
6 10 No
6 13 No
6 15 No
6 48 No
6 49 No
6 60 No
6 61 No
6 71 Yes
6 72 Yes
6 80

ValueError: ('daxday\\n\\nStep-by-step explanation: The sentence structure indicates that daxday happened first, leading to blicketbash as a consequence. The word "consequently" shows cause and effect, so daxday started first.\')', 'daxday  step-by-step explanation: the sentence structure indicates that daxday happened first, leading to blicketbash as a consequence. the word "consequently" shows cause and effect, so daxday started first.')

In [109]:
path = '../data/results/lrm/grounded_deepseek-ai_DeepSeek-R1-Distill-Qwen-14B_responses_.csv'
grounded = utils.read_csv_dict(path)

rows = []
for i, response in enumerate(grounded):
    ans = get_answer(response['idx'])
    ans = ans.replace("\\", '')
    ans = ans.replace("}", '')
    
    row = [i, 1, ans, True]
    rows.append(row)

header = ["idx","prob","label","entailed"]
out = '../data/results/all/lrm_grounded.csv'

utils.write_csv(rows, out, header = header)